## Initialize Qdrant

In [2]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

/Users/khoa/Agent_practice/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client import QdrantClient, models 
from langchain_openai import OpenAIEmbeddings
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")


dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
qdrant = QdrantVectorStore.from_existing_collection(
    collection_name="document_collection",
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    retrieval_mode = RetrievalMode.HYBRID,
    vector_name = "dense",
    sparse_vector_name = "sparse"
    )

In [11]:
qdrant.similarity_search("What is Home Credit?, k=1")

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-02-02T00:41:51+00:00', 'source': '/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/tmppep5_hzk.pdf', 'file_path': '/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/tmppep5_hzk.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-02-02T00:41:51+00:00', 'trapped': '', 'modDate': 'D:20260202004151Z', 'creationDate': 'D:20260202004151Z', 'page': 0, '_id': 'f5f538b7-78e5-4512-9572-dd1db0cae57b', '_collection_name': 'document_collection'}, page_content='Khoa Bui\n\x83 (437) 443-7831\n# bndk2108@gmail.com\nï linkedin.com/in/bndk2108\n§ github.com/hyolee1999\nEducation\nHo Chi Minh University of Technology\nOct 2017 – April 2022\nBachelor of Computer Science\nHo Chi Minh City, Vietnam\n• Thesis: Develop Android Application for Removing Unwanted Objects. York University\nSep 2025 – Present\nMaster of Computer Science\nToront

In [21]:
def get_all_documents(
    client: QdrantClient,
    collection_name: str,
    batch_size: int = 100,
) -> list:
    all_points = []
    offset = None

    while True:
        points, next_offset = client.scroll(
            collection_name=collection_name,
            limit=batch_size,
            offset=offset,
            with_payload=True,
            with_vectors=False,
        )

        all_points.extend(points)

        if next_offset is None:
            break

        offset = next_offset

    return all_points

documents = get_all_documents(client, "document_collection", batch_size=100)
print(f"🗂️  Total documents indexed: {len(documents)}")



🗂️  Total documents indexed: 26


In [22]:
# 3️⃣  Show a few samples (ID + stored payload, e.g., the original chunk text)
for point in documents:          # show first 5 records
    print("-" * 40)
    print(f"ID: {point.id}")
    # payload is a dict; most often it contains the original text under a key like "text"
    for key, value in point.payload.items():
        print(f"{key}: {value}")

----------------------------------------
ID: 01db4822-ffcd-42d7-8800-44ef48e8c714
page_content: personally, contributing to my later career as data engineer specializing in enhancing 
large-scale AI systems who not just come up with solutions theoretically but can make it 
understandable to non-technical users and applicable to practical business situations. Thank you for taking time to consider my application, 
Bui Ngoc Dang Khoa.
metadata: {'producer': 'Skia/PDF m133 Google Docs Renderer', 'creator': '', 'creationdate': '', 'source': '/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/tmp8g8cqyvo.pdf', 'file_path': '/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/tmp8g8cqyvo.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'Statement_Of_Interest_tmu_2', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 1}
----------------------------------------
ID: 0c9bf82b-8754-4b90-9f5c-bc6bb36dd1c2
page_content: Guanghui (Ric

In [5]:
# from langchain_openai import OpenAIEmbeddings
# dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
# print(dense_embeddings)
# print(type(dense_embeddings))
# print(dense_embeddings.model)
# print(dense_embeddings.model_kwargs)
# print(dense_embeddings._invocation_params)

# print(
#     len(
#         dense_embeddings.embed_documents(
#             ["dummy_text"]
#         )[0]
#     )
# )

client.delete_collection(collection_name="document_collection")

True

## Create a collection

In [2]:
from qdrant_client.models import Distance, VectorParams


client.recreate_collection(
    collection_name="my_collection",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE)
)


/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/ipykernel_30273/753245820.py:4: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

## Add vectors

In [3]:
from qdrant_client.models import PointStruct

operation = client.upsert(
    collection_name="my_collection",
    points=[
        PointStruct(
            id=1,
            vector=[0.1] * 1536,
            payload={"text": "This is a sample document."}
        )
    ]
)


In [4]:
client.upsert(
    collection_name="my_collection",
    points=[
            PointStruct(
                id=1,
                vector=[0.2] * 1536,
                payload={"text": "This is a sample document."}
            ),
            PointStruct(
                id=2,
                vector=[0.3] * 1536,
                payload={"text": "This is another sample document."}
            ),
            PointStruct(
                id=3,
                vector=[0.4] * 1536,
                payload={"text": "This is a third sample document."}
            )
    ]
)

UpdateResult(operation_id=6, status=<UpdateStatus.COMPLETED: 'completed'>)

In [13]:
result =client.query_points(
    collection_name="my_collection",
    query=[0.1] * 1536,
    limit=5,
    with_payload=True
    # query_text  ="Hello world"
)

In [15]:
for point in result.points:
    print(f"ID: {point.id}, Score: {point.score}, Payload: {point.payload}")

ID: 3, Score: 1.0000015, Payload: {'text': 'This is a third sample document.'}
ID: 1, Score: 1.0000015, Payload: {'text': 'This is a sample document.'}
ID: 2, Score: 1.0000007, Payload: {'text': 'This is another sample document.'}


## BM25 vs dense search

BM25 is useful when exact words matter, such as product names, error codes, model numbers, or part numbers. Dense search is better when you care more about meaning than exact keywords.

In [20]:
from fastembed.sparse.sparse_text_embedding import SparseTextEmbedding
from qdrant_client.models import SparseVector

# 1) Dense search: semantic similarity with a vector query.
dense_hits = client.query_points(
    collection_name="my_collection",
    query=[0.1] * 1536,
    limit=3,
    with_payload=True,
)

print("Dense search results:")
for point in dense_hits.points:
    print(f"ID: {point.id}, Score: {point.score}, Payload: {point.payload}")

# 2) BM25 search: exact keyword matching on a sparse text collection.
client.set_sparse_model("Qdrant/bm25")

bm25_collection = "bm25_demo"
client.add(
    collection_name=bm25_collection,
    documents=[
        "Apple iPhone 15 Pro Max 256GB",
        "Samsung Galaxy S24 Ultra 256GB",
        "Apple MacBook Air M3",
    ],
)

bm25_model = SparseTextEmbedding("Qdrant/bm25")
bm25_query = list(bm25_model.query_embed("iphone 256gb"))[0]

bm25_hits = client.query_points(
    collection_name=bm25_collection,
    query=SparseVector(
        indices=bm25_query.indices.tolist(),
        values=bm25_query.values.tolist(),
    ),
    using=client.get_sparse_vector_field_name(),
    limit=3,
    with_payload=True,
)

print("\nBM25 search results:")
for point in bm25_hits.points:
    print(f"ID: {point.id}, Score: {point.score}, Payload: {point.payload}")

Dense search results:
ID: 3, Score: 1.0000015, Payload: {'text': 'This is a third sample document.'}
ID: 1, Score: 1.0000015, Payload: {'text': 'This is a sample document.'}
ID: 2, Score: 1.0000007, Payload: {'text': 'This is another sample document.'}

BM25 search results:
ID: 9dd20a04-519e-4336-a72c-9982c77fa0d6, Score: 2.4503899, Payload: {'document': 'Apple iPhone 15 Pro Max 256GB'}
ID: 2edc37e6-009c-4fa9-94e7-5f94ef5c3a03, Score: 2.4503899, Payload: {'document': 'Apple iPhone 15 Pro Max 256GB'}
ID: 9340239e-d193-4bf9-97b9-a92e2626fba2, Score: 0.73774153, Payload: {'document': 'Samsung Galaxy S24 Ultra 256GB'}


In [25]:
import time
from fastembed import TextEmbedding
from fastembed.sparse.sparse_text_embedding import SparseTextEmbedding
from qdrant_client.models import PointStruct, SparseVector, VectorParams, Distance

# Same dataset, two collections: one dense, one sparse/BM25.
comparison_docs = [
    {"id": 1, "text": "Apple iPhone 15 Pro Max 256GB"},
    {"id": 2, "text": "Samsung Galaxy S24 Ultra 256GB"},
    {"id": 3, "text": "Apple MacBook Air M3"},
    {"id": 4, "text": "Android phone with 256GB storage"},
]

queries = [
    ("iphone 256gb", 1),
    ("apple laptop m3", 3),
    ("galaxy ultra", 2),
]

# Dense collection
client.recreate_collection(
    collection_name="dense_compare",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

dense_model = TextEmbedding()

dense_points = []
for item in comparison_docs:
    dense_vector = list(dense_model.query_embed(item["text"]))[0].tolist()
    dense_points.append(
        PointStruct(
            id=item["id"],
            vector=dense_vector,
            payload={"text": item["text"]},
        )
    )

client.upsert(collection_name="dense_compare", points=dense_points)

# Sparse/BM25 collection
client.set_sparse_model("Qdrant/bm25")
client.recreate_collection(
    collection_name="bm25_compare",
    sparse_vectors_config={
        client.get_sparse_vector_field_name(): client.get_fastembed_sparse_vector_params()[
            client.get_sparse_vector_field_name()
        ]
    },
)

bm25_model = SparseTextEmbedding("Qdrant/bm25")

bm25_points = []
for item in comparison_docs:
    sparse_vector = list(bm25_model.query_embed(item["text"]))[0]
    bm25_points.append(
        PointStruct(
            id=item["id"],
            vector={
                client.get_sparse_vector_field_name(): SparseVector(
                    indices=sparse_vector.indices.tolist(),
                    values=sparse_vector.values.tolist(),
                )
            },
            payload={"text": item["text"]},
        )
    )

client.upsert(collection_name="bm25_compare", points=bm25_points)

# Compare top-1 hit rate and elapsed time for the same queries.
def evaluate_collection(collection_name: str, query_builder, using: str | None = None):
    hits = 0
    start_time = time.perf_counter()

    for query_text, expected_id in queries:
        query_value = query_builder(query_text)
        response = client.query_points(
            collection_name=collection_name,
            query=query_value,
            using=using,
            limit=1,
            with_payload=True,
        )
        if response.points and response.points[0].id == expected_id:
            hits += 1

    elapsed_seconds = time.perf_counter() - start_time
    hit_rate = hits / len(queries)
    avg_ms = (elapsed_seconds / len(queries)) * 1000
    return hit_rate, elapsed_seconds, avg_ms

# Dense search
# This measures the full query flow, including embedding the query text.
dense_hit_rate, dense_elapsed, dense_avg_ms = evaluate_collection(
    "dense_compare",
    query_builder=lambda text: list(dense_model.query_embed(text))[0].tolist(),
    using=None,
)

# BM25 search
bm25_hit_rate, bm25_elapsed, bm25_avg_ms = evaluate_collection(
    "bm25_compare",
    query_builder=lambda text: SparseVector(
        indices=(bm25_vec := list(bm25_model.query_embed(text))[0]).indices.tolist(),
        values=bm25_vec.values.tolist(),
    ),
    using=client.get_sparse_vector_field_name(),
)

print(f"Dense top-1 hit rate: {dense_hit_rate:.2f}")
print(f"Dense total time:     {dense_elapsed:.4f} s")
print(f"Dense avg/query:      {dense_avg_ms:.2f} ms")
print(f"BM25 top-1 hit rate:   {bm25_hit_rate:.2f}")
print(f"BM25 total time:       {bm25_elapsed:.4f} s")
print(f"BM25 avg/query:        {bm25_avg_ms:.2f} ms")

/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/ipykernel_14659/614330401.py:21: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Dense top-1 hit rate: 1.00
Dense total time:     0.0177 s
Dense avg/query:      5.90 ms
BM25 top-1 hit rate:   1.00
BM25 total time:       0.0082 s
BM25 avg/query:        2.73 ms


/var/folders/yz/96nfmvzs2q934d80wkdh280c0000gn/T/ipykernel_14659/614330401.py:43: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(
